In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import sklearn

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, ConfusionMatrixDisplay,
    average_precision_score
)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA


# FILE PATHS

ACTIVITY_FILE = "customer_churn_data.xlsx"
DEMO_FILE     = "demographic_data.csv"
PAYMENT_FILE  = "subscription_payment_history.csv"
OUTPUT_DIR    = "./"          # folder where PNGs + CSV are saved


# GLOBAL STYLE

PALETTE = {
    "primary":   "#6C63FF",
    "secondary": "#FF6584",
    "accent":    "#43B89C",
    "warn":      "#F7B731",
    "dark":      "#2D2D2D",
    "light":     "#F5F5F5",
    "mid":       "#B0B0B0",
}
plt.rcParams.update({
    "figure.facecolor":  PALETTE["light"],
    "axes.facecolor":    "#FFFFFF",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "axes.labelsize":    11,
    "axes.titlesize":    13,
    "axes.titleweight":  "bold",
    "xtick.labelsize":   9,
    "ytick.labelsize":   9,
    "legend.fontsize":   9,
    "font.family":       "DejaVu Sans",
})



# DATA LOADING & MERGING

print("\n" + "="*60)
print("DATA LOADING & MERGING")
print("="*60)

activity = pd.read_excel(ACTIVITY_FILE)
demo     = pd.read_csv(DEMO_FILE)
payment  = pd.read_csv(PAYMENT_FILE)

print(f"Activity logs   : {activity.shape}")
print(f"Demographic     : {demo.shape}")
print(f"Payment history : {payment.shape}")



# FEATURE ENGINEERING  (aggregate to one row per customer)

print("\n" + "="*60)
print("FEATURE ENGINEERING")
print("="*60)

agg = activity.groupby("CustomerID").agg(
    TotalLogins        = ("Login",            "sum"),
    AvgDailyLogins     = ("Login",            "mean"),
    TotalWatchMins     = ("WatchTimeMinutes",  "sum"),
    AvgWatchMins       = ("WatchTimeMinutes",  "mean"),
    ActiveDays         = ("Date",              "count"),
    UniqueContentTypes = ("ContentPreference", "nunique"),
    MostWatchedContent = ("ContentPreference", lambda x: x.mode()[0]),
    SubscriptionPlan   = ("SubscriptionPlan",  "first"),
    AgeGroup           = ("AgeGroup",          "first"),
    Churned            = ("Churned",           "max"),
).reset_index()

# Composite engagement score  (0–100)
agg["EngagementScore"] = (
    0.5 * (agg["TotalLogins"]   / agg["TotalLogins"].max()) +
    0.5 * (agg["TotalWatchMins"] / agg["TotalWatchMins"].max())
) * 100

# Merge all sources
df = agg.merge(demo, on="CustomerID", how="left")
df = df.merge(payment[["CustomerID", "BillingCycle", "IsAutoRenew"]],
              on="CustomerID", how="left")

df["IsMonthly"]   = (df["BillingCycle"] == "Monthly").astype(int)
df["NoAutoRenew"] = (~df["IsAutoRenew"]).astype(int)

print(f"Master dataset  : {df.shape}")
print(f"Churn rate      : {df['Churned'].mean():.2%}")



# EDA DASHBOARD

print("\n" + "="*60)
print("EDA DASHBOARD")
print("="*60)

fig = plt.figure(figsize=(20, 16), facecolor=PALETTE["light"])
fig.suptitle("CHURNLYTICS — EDA Dashboard | StreamNow Pvt. Ltd.",
             fontsize=18, fontweight="bold", color=PALETTE["dark"], y=0.98)
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# Churn donut
ax = fig.add_subplot(gs[0, 0])
vals = [len(df) - df["Churned"].sum(), df["Churned"].sum()]
ax.pie(vals, labels=["Retained", "Churned"],
       colors=[PALETTE["accent"], PALETTE["secondary"]],
       autopct="%1.1f%%", startangle=90,
       wedgeprops=dict(width=0.55, edgecolor="white", linewidth=2))
ax.set_title("Overall Churn Rate")

# Churn by plan
ax = fig.add_subplot(gs[0, 1])
plan_churn = df.groupby("SubscriptionPlan")["Churned"].mean().sort_values(ascending=False)
cols = [PALETTE["secondary"] if v == plan_churn.max() else PALETTE["primary"]
        for v in plan_churn.values]
bars = ax.bar(plan_churn.index, plan_churn.values * 100, color=cols, edgecolor="white", zorder=2)
ax.yaxis.grid(True, linestyle="--", alpha=0.5, zorder=1)
ax.set_ylabel("Churn Rate (%)")
ax.set_title("Churn Rate by Plan")
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.5,
            f"{b.get_height():.1f}%", ha="center", fontsize=9, fontweight="bold")

# Churn by age group
ax = fig.add_subplot(gs[0, 2])
age_churn = df.groupby("AgeGroup")["Churned"].mean().sort_values(ascending=False)
ax.barh(age_churn.index, age_churn.values * 100,
        color=PALETTE["warn"], edgecolor="white")
ax.xaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_xlabel("Churn Rate (%)")
ax.set_title("Churn Rate by Age Group")

# Engagement score distribution
ax = fig.add_subplot(gs[1, 0])
ax.hist(df[df["Churned"]==False]["EngagementScore"], bins=30,
        alpha=0.7, color=PALETTE["accent"], label="Retained", density=True)
ax.hist(df[df["Churned"]==True]["EngagementScore"],  bins=30,
        alpha=0.7, color=PALETTE["secondary"], label="Churned",   density=True)
ax.set_xlabel("Engagement Score")
ax.set_ylabel("Density")
ax.set_title("Engagement Score by Churn")
ax.legend()

# Watch time vs logins scatter
ax = fig.add_subplot(gs[1, 1])
sample = df.sample(min(250, len(df)), random_state=42)
sc_colors = [PALETTE["secondary"] if c else PALETTE["accent"] for c in sample["Churned"]]
ax.scatter(sample["TotalLogins"], sample["TotalWatchMins"],
           c=sc_colors, alpha=0.6, s=35, edgecolors="white", linewidths=0.5)
ax.set_xlabel("Total Logins")
ax.set_ylabel("Total Watch Mins")
ax.set_title("Logins vs Watch Time")
ax.legend(handles=[mpatches.Patch(color=PALETTE["accent"],    label="Retained"),
                   mpatches.Patch(color=PALETTE["secondary"], label="Churned")])

# Content preference
ax = fig.add_subplot(gs[1, 2])
content_churn = df.groupby("MostWatchedContent")["Churned"].mean().sort_values()
ax.barh(content_churn.index, content_churn.values * 100,
        color=PALETTE["primary"], edgecolor="white")
ax.xaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_xlabel("Churn Rate (%)")
ax.set_title("Churn by Content Preference")

# Billing cycle
ax = fig.add_subplot(gs[2, 0])
billing_churn = df.groupby("BillingCycle")["Churned"].mean().sort_values(ascending=False)
ax.bar(billing_churn.index, billing_churn.values * 100,
       color=[PALETTE["secondary"], PALETTE["warn"], PALETTE["primary"]],
       edgecolor="white")
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_ylabel("Churn Rate (%)")
ax.set_title("Churn by Billing Cycle")

# Location
ax = fig.add_subplot(gs[2, 1])
loc_churn = df.groupby("Location")["Churned"].mean().sort_values(ascending=False)
ax.bar(loc_churn.index, loc_churn.values * 100,
       color=PALETTE["accent"], edgecolor="white")
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_ylabel("Churn Rate (%)")
ax.set_title("Churn by Location")

# Auto-renew
ax = fig.add_subplot(gs[2, 2])
ar_churn = df.groupby("IsAutoRenew")["Churned"].mean()
ar_churn.index = ["No Auto-Renew", "Auto-Renew"]
ax.bar(ar_churn.index, ar_churn.values * 100,
       color=[PALETTE["secondary"], PALETTE["accent"]],
       edgecolor="white", width=0.5)
ax.yaxis.grid(True, linestyle="--", alpha=0.5)
ax.set_ylabel("Churn Rate (%)")
ax.set_title("Churn by Auto-Renew Status")

plt.savefig(OUTPUT_DIR + "1_eda_dashboard.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ 1_eda_dashboard.png")



# FEATURE PREP FOR ML

print("\n" + "="*60)
print("ML FEATURE PREPARATION")
print("="*60)

FEATURES = [
    "TotalLogins", "AvgDailyLogins", "TotalWatchMins", "AvgWatchMins",
    "ActiveDays", "UniqueContentTypes", "EngagementScore", "HouseholdSize",
    "IsMonthly", "NoAutoRenew",
    "SubscriptionPlan", "AgeGroup", "MostWatchedContent", "Location", "PrimaryDevice",
]
TARGET = "Churned"

ml_df = df[FEATURES + [TARGET]].copy()
cat_cols = ["SubscriptionPlan", "AgeGroup", "MostWatchedContent", "Location", "PrimaryDevice"]
for col in cat_cols:
    ml_df[col] = LabelEncoder().fit_transform(ml_df[col].astype(str))
ml_df[TARGET] = ml_df[TARGET].astype(int)

X = ml_df[FEATURES]
y = ml_df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)
X_test_sc   = scaler.transform(X_test)

print(f"  Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"  Churn rate  train={y_train.mean():.2%}  test={y_test.mean():.2%}")



# TRAIN FOUR MODELS

print("\n" + "="*60)
print("  STEP 5: MODEL TRAINING")
print("="*60)

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", C=0.5, random_state=42),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=10, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=8, min_samples_leaf=5,
        class_weight="balanced", random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=150, learning_rate=0.08, max_depth=5,
        subsample=0.8, random_state=42),
}

results = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    X_tr = X_train_sc if name == "Logistic Regression" else X_train
    X_te = X_test_sc  if name == "Logistic Regression" else X_test

    model.fit(X_tr, y_train)
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    rep     = classification_report(y_test, y_pred, output_dict=True)
    cv_roc  = cross_val_score(
        model, X_train_sc if name == "Logistic Regression" else X_train,
        y_train, cv=cv, scoring="roc_auc")

    results[name] = {
        "model":     model,
        "y_pred":    y_pred,
        "y_proba":   y_proba,
        "roc_auc":   roc_auc_score(y_test, y_proba),
        "avg_prec":  average_precision_score(y_test, y_proba),
        "cv_roc":    cv_roc.mean(),
        "cv_std":    cv_roc.std(),
        "precision": rep["1"]["precision"],
        "recall":    rep["1"]["recall"],
        "f1":        rep["1"]["f1-score"],
        "X_te":      X_te,
    }
    print(f"  {name:25s}  ROC-AUC={results[name]['roc_auc']:.3f}"
          f"  F1={results[name]['f1']:.3f}"
          f"  CV={results[name]['cv_roc']:.3f}±{results[name]['cv_std']:.3f}")

best_name = max(results, key=lambda k: results[k]["roc_auc"])
best      = results[best_name]
print(f"\n  ★ Best model: {best_name}  (AUC={best['roc_auc']:.3f})")



# MODEL EVALUATION DASHBOARD

print("\n" + "="*60)
print("MODEL EVALUATION DASHBOARD")
print("="*60)

model_colors = {
    "Logistic Regression": PALETTE["primary"],
    "Decision Tree":       PALETTE["warn"],
    "Random Forest":       PALETTE["accent"],
    "Gradient Boosting":   PALETTE["secondary"],
}

fig, axes = plt.subplots(2, 3, figsize=(20, 12), facecolor=PALETTE["light"])
fig.suptitle("CHURNLYTICS — Model Evaluation Dashboard",
             fontsize=17, fontweight="bold", color=PALETTE["dark"])

# 6.1  ROC curves
ax = axes[0, 0]
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res["y_proba"])
    ax.plot(fpr, tpr, lw=2, color=model_colors[name],
            label=f"{name} ({res['roc_auc']:.3f})")
ax.plot([0,1],[0,1], "k--", lw=1, alpha=0.4)
ax.set(xlabel="FPR", ylabel="TPR", title="ROC Curves")
ax.legend(loc="lower right", fontsize=8)

# Precision-recall curves
ax = axes[0, 1]
for name, res in results.items():
    prec, rec, _ = precision_recall_curve(y_test, res["y_proba"])
    ax.plot(rec, prec, lw=2, color=model_colors[name],
            label=f"{name} (AP={res['avg_prec']:.3f})")
ax.set(xlabel="Recall", ylabel="Precision", title="Precision-Recall Curves")
ax.legend(fontsize=8)

# Metric bar comparison
ax = axes[0, 2]
metrics_df = pd.DataFrame({
    n: {"ROC-AUC": r["roc_auc"], "Precision": r["precision"],
        "Recall":  r["recall"],   "F1":        r["f1"]}
    for n, r in results.items()
}).T
x, w = np.arange(len(metrics_df)), 0.2
for i, (met, col) in enumerate(zip(
        ["ROC-AUC","Precision","Recall","F1"],
        [PALETTE["primary"], PALETTE["accent"], PALETTE["warn"], PALETTE["secondary"]])):
    ax.bar(x + i*w, metrics_df[met], width=w, label=met, color=col, alpha=0.85, edgecolor="white")
ax.set_xticks(x + w*1.5)
ax.set_xticklabels([n.replace(" ","\n") for n in metrics_df.index], fontsize=8)
ax.set_ylim(0, 1.1)
ax.yaxis.grid(True, linestyle="--", alpha=0.4)
ax.legend(fontsize=8)
ax.set_title("Model Metric Comparison")

# Confusion matrix (best model)
ax = axes[1, 0]
ConfusionMatrixDisplay(
    confusion_matrix(y_test, best["y_pred"]),
    display_labels=["Retained","Churned"]
).plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title(f"Confusion Matrix — {best_name}")

# Feature importances (best tree model)
ax = axes[1, 1]
tree_name = max(
    {k: v for k, v in results.items() if k in ("Random Forest","Gradient Boosting")},
    key=lambda k: results[k]["roc_auc"])
feat_imp = pd.Series(
    results[tree_name]["model"].feature_importances_, index=FEATURES
).sort_values().tail(12)
fi_colors = [PALETTE["secondary"] if v == feat_imp.max() else PALETTE["primary"]
             for v in feat_imp.values]
ax.barh(feat_imp.index, feat_imp.values, color=fi_colors, edgecolor="white")
ax.xaxis.grid(True, linestyle="--", alpha=0.4)
ax.set_title(f"Feature Importances — {tree_name}")

# CV stability
ax = axes[1, 2]
names     = list(results.keys())
cv_scores = [results[n]["cv_roc"] for n in names]
cv_stds   = [results[n]["cv_std"] for n in names]
bars = ax.bar(range(len(names)), cv_scores,
              yerr=cv_stds, color=[model_colors[n] for n in names],
              edgecolor="white", capsize=6, zorder=2)
ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.replace(" ","\n") for n in names], fontsize=8)
ax.set_ylim(0.5, 1.0)
ax.yaxis.grid(True, linestyle="--", alpha=0.4, zorder=1)
ax.set_ylabel("5-Fold CV ROC-AUC")
ax.set_title("Cross-Validation Stability")
for b, s in zip(bars, cv_scores):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.005,
            f"{s:.3f}", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR + "2_model_evaluation.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ 2_model_evaluation.png")



# RISK-SCORE ALL CUSTOMERS

print("\n" + "="*60)
print("RISK SCORING")
print("="*60)

X_all = ml_df[FEATURES].copy()
if best_name == "Logistic Regression":
    churn_proba = best["model"].predict_proba(scaler.transform(X_all))[:, 1]
else:
    churn_proba = best["model"].predict_proba(X_all)[:, 1]

df["ChurnProbability"] = churn_proba
df["RiskTier"] = pd.cut(
    churn_proba, bins=[0, 0.30, 0.55, 0.75, 1.0],
    labels=["Low Risk","Medium Risk","High Risk","Critical Risk"])

print(df.groupby("RiskTier", observed=True).agg(
    Customers=("CustomerID","count"),
    AvgChurnProb=("ChurnProbability","mean"),
    ActualChurned=("Churned","sum")
).to_string())



# K-MEANS CUSTOMER SEGMENTATION

print("\n" + "="*60)
print("CUSTOMER SEGMENTATION")
print("="*60)

seg_features = ["EngagementScore","TotalWatchMins","TotalLogins",
                "AvgWatchMins","ChurnProbability"]
seg_scaled = StandardScaler().fit_transform(df[seg_features].fillna(0))

# Elbow to validate k=4
inertias = []
for k in range(2, 9):
    inertias.append(KMeans(n_clusters=k, random_state=42, n_init=10).fit(seg_scaled).inertia_)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df["Segment"] = kmeans.fit_predict(seg_scaled)

seg_profile = df.groupby("Segment").agg(
    Count         =("CustomerID",       "count"),
    AvgEngagement =("EngagementScore",  "mean"),
    AvgChurnProb  =("ChurnProbability", "mean"),
    Churned       =("Churned",          "sum"),
).reset_index()

# Auto-label segments
used, final_labels = set(), {}
order = [
    ("Champions",       seg_profile.loc[seg_profile["AvgEngagement"].idxmax(), "Segment"]),
    ("High-Risk",       seg_profile.loc[seg_profile["AvgChurnProb"].idxmax(),  "Segment"]),
    ("At-Risk Dormant", seg_profile.loc[seg_profile["AvgEngagement"].idxmin(), "Segment"]),
]
for label, seg in order:
    if seg not in final_labels and label not in used:
        final_labels[seg] = label; used.add(label)
for seg in seg_profile["Segment"]:
    if seg not in final_labels:
        final_labels[seg] = "Loyal Engaged"

df["SegmentLabel"]     = df["Segment"].map(final_labels)
seg_profile["Label"]   = seg_profile["Segment"].map(final_labels)
print(seg_profile[["Label","Count","AvgEngagement","AvgChurnProb","Churned"]].to_string(index=False))



# SEGMENTATION DASHBOARD

print("\n" + "="*60)
print("SEGMENTATION DASHBOARD")
print("="*60)

pca        = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(seg_scaled)

seg_color_map = {
    "Champions":       PALETTE["accent"],
    "Loyal Engaged":   PALETTE["primary"],
    "High-Risk":       PALETTE["secondary"],
    "At-Risk Dormant": PALETTE["warn"],
}

fig, axes = plt.subplots(2, 3, figsize=(20, 12), facecolor=PALETTE["light"])
fig.suptitle("CHURNLYTICS — Segmentation & Risk Dashboard",
             fontsize=17, fontweight="bold", color=PALETTE["dark"])

# PCA scatter
ax = axes[0, 0]
for label, color in seg_color_map.items():
    mask = df["SegmentLabel"] == label
    ax.scatter(pca_coords[mask, 0], pca_coords[mask, 1],
               c=color, label=label, alpha=0.6, s=25)
ax.set(xlabel=f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)",
       ylabel=f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)",
       title="Customer Segments (PCA)")
ax.legend(fontsize=8)

# Segment profile bars
ax = axes[0, 1]
sp = seg_profile.set_index("Label")[["AvgEngagement","AvgChurnProb"]].copy()
sp["AvgChurnProb"] *= 100
x_pos = np.arange(len(sp)); w = 0.35
ax.bar(x_pos - w/2, sp["AvgEngagement"], width=w,
       color=PALETTE["accent"],    label="Avg Engagement", alpha=0.85)
ax.bar(x_pos + w/2, sp["AvgChurnProb"],  width=w,
       color=PALETTE["secondary"], label="Avg Churn Prob %", alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels(sp.index, fontsize=8, rotation=15)
ax.yaxis.grid(True, linestyle="--", alpha=0.4)
ax.legend(fontsize=8)
ax.set_title("Segment Profile Comparison")

# Risk tier donut
ax = axes[0, 2]
risk_colors  = {"Low Risk": PALETTE["accent"], "Medium Risk": PALETTE["warn"],
                "High Risk": PALETTE["primary"], "Critical Risk": PALETTE["secondary"]}
risk_counts  = df["RiskTier"].value_counts().sort_index()
ax.pie(risk_counts.values,
       labels=risk_counts.index,
       colors=[risk_colors.get(k, PALETTE["mid"]) for k in risk_counts.index],
       autopct="%1.1f%%", startangle=90,
       wedgeprops=dict(width=0.6, edgecolor="white", linewidth=2))
ax.set_title("Risk Tier Distribution")

# Churn probability histogram
ax = axes[1, 0]
ax.hist(df["ChurnProbability"], bins=40,
        color=PALETTE["primary"], edgecolor="white", linewidth=0.5)
ax.axvline(0.5, color=PALETTE["secondary"], linestyle="--", lw=2, label="50% threshold")
ax.set(xlabel="Churn Probability", ylabel="# Customers",
       title="Churn Probability Distribution")
ax.legend()

# Elbow curve
ax = axes[1, 1]
ax.plot(range(2, 9), inertias, marker="o", color=PALETTE["primary"],
        lw=2, markersize=7, markerfacecolor=PALETTE["secondary"],
        markeredgecolor="white")
ax.axvline(4, color=PALETTE["warn"], linestyle="--", lw=2, label="k=4 selected")
ax.set(xlabel="k", ylabel="Inertia", title="K-Means Elbow Curve")
ax.legend(); ax.yaxis.grid(True, linestyle="--", alpha=0.4)

# Boxplot per segment
ax = axes[1, 2]
seg_order = ["Champions","Loyal Engaged","High-Risk","At-Risk Dormant"]
bp = ax.boxplot([df[df["SegmentLabel"]==s]["ChurnProbability"].values for s in seg_order],
                patch_artist=True,
                medianprops=dict(color="white", lw=2))
for patch, s in zip(bp["boxes"], seg_order):
    patch.set_facecolor(seg_color_map[s]); patch.set_alpha(0.8)
ax.set_xticks(range(1, 5))
ax.set_xticklabels(seg_order, fontsize=8, rotation=10)
ax.set(ylabel="Churn Probability", title="Churn Probability by Segment")
ax.yaxis.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(OUTPUT_DIR + "3_segmentation_dashboard.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ 3_segmentation_dashboard.png")



# RETENTION STRATEGY SIMULATION

print("\n" + "="*60)
print("RETENTION SIMULATION")
print("="*60)

PLAN_REVENUE = {"Basic": 199, "Standard": 399, "Premium": 699}
df["MonthlyRevenue"] = df["SubscriptionPlan"].map(PLAN_REVENUE).fillna(199)
df["CLV_12M"]        = df["MonthlyRevenue"] * 12 * (1 - df["ChurnProbability"])

baseline_churn_rate = df["Churned"].mean()

campaigns = {
    "High-Risk": {
        "strategy":        "Personalized Win-Back Offer",
        "action":          "30% discount + content unlock",
        "cost_per_cust":   50,
        "churn_reduction": 0.35,
    },
    "At-Risk Dormant": {
        "strategy":        "Re-Engagement Nudge",
        "action":          "Curated watchlist push notification",
        "cost_per_cust":   15,
        "churn_reduction": 0.20,
    },
    "Champions": {
        "strategy":        "Loyalty Reward",
        "action":          "Referral bonus + early feature access",
        "cost_per_cust":   25,
        "churn_reduction": 0.10,
    },
    "Loyal Engaged": {
        "strategy":        "Upgrade Incentive",
        "action":          "Next tier at 20% off for 3 months",
        "cost_per_cust":   20,
        "churn_reduction": 0.15,
    },
}

campaign_results = []
for seg, cfg in campaigns.items():
    seg_df         = df[df["SegmentLabel"] == seg]
    n              = len(seg_df)
    avg_churn_prob = seg_df["ChurnProbability"].mean()
    expected_save  = int(n * avg_churn_prob * cfg["churn_reduction"])
    revenue_saved  = expected_save * seg_df["MonthlyRevenue"].mean() * 3
    campaign_cost  = n * cfg["cost_per_cust"]
    net_benefit    = revenue_saved - campaign_cost
    roi            = (net_benefit / campaign_cost * 100) if campaign_cost > 0 else 0

    campaign_results.append({
        "Segment":        seg,
        "Strategy":       cfg["strategy"],
        "Customers":      n,
        "Expected Saves": expected_save,
        "Campaign Cost":  f"₹{campaign_cost:,.0f}",
        "Revenue Saved":  f"₹{revenue_saved:,.0f}",
        "Net Benefit":    f"₹{net_benefit:,.0f}",
        "ROI %":          f"{roi:.0f}%",
    })
    print(f"  [{seg:16s}]  saves {expected_save:3d} customers  |  ROI = {roi:.0f}%")

total_saves         = sum(r["Expected Saves"] for r in campaign_results)
new_churn_rate      = max(0, baseline_churn_rate - total_saves / len(df))
churn_reduction_pct = (baseline_churn_rate - new_churn_rate) / baseline_churn_rate * 100
print(f"\n  Baseline churn : {baseline_churn_rate:.2%}")
print(f"  Post-campaign  : {new_churn_rate:.2%}")
print(f"  Reduction      : {churn_reduction_pct:.1f}%  (target 15%)")



# EXECUTIVE SUMMARY DASHBOARD

print("\n" + "="*60)
print("EXECUTIVE SUMMARY DASHBOARD")
print("="*60)

fig = plt.figure(figsize=(20, 14), facecolor=PALETTE["dark"])
fig.suptitle("CHURNLYTICS — Executive Summary | StreamNow Pvt. Ltd.",
             fontsize=20, fontweight="bold", color="white", y=0.98)
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.5, wspace=0.4)

# KPI cards
kpis = [
    ("Total Customers",     f"{len(df):,}",               PALETTE["primary"]),
    ("Baseline Churn Rate", f"{baseline_churn_rate:.1%}",  PALETTE["secondary"]),
    ("Best Model AUC",      f"{best['roc_auc']:.3f}",      PALETTE["accent"]),
    ("Churn Reduction",     f"{churn_reduction_pct:.1f}%", PALETTE["warn"]),
]
for i, (label, value, color) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor(color); ax.axis("off")
    ax.text(0.5, 0.62, value, ha="center", fontsize=26,
            fontweight="bold", color="white", transform=ax.transAxes)
    ax.text(0.5, 0.28, label, ha="center", fontsize=11,
            color="white", alpha=0.85, transform=ax.transAxes)

# Campaign saves bar
ax = fig.add_subplot(gs[1, 0:2])
ax.set_facecolor("#1a1a2e")
bars = ax.bar(
    [r["Segment"] for r in campaign_results],
    [r["Expected Saves"] for r in campaign_results],
    color=[seg_color_map.get(r["Segment"], PALETTE["mid"]) for r in campaign_results],
    edgecolor="#333", zorder=2)
ax.yaxis.grid(True, linestyle="--", alpha=0.2, color="white", zorder=1)
ax.set_ylabel("Customers Saved", color="white")
ax.set_title("Expected Saves by Campaign", color="white")
ax.tick_params(colors="white")
for b in bars:
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.1,
            f"{int(b.get_height())}", ha="center", color="white",
            fontsize=10, fontweight="bold")
for spine in ax.spines.values(): spine.set_color("#444")

# Churn trajectory
ax = fig.add_subplot(gs[1, 2:4])
ax.set_facecolor("#1a1a2e")
months = ["Month 0\n(Baseline)", "Month 1", "Month 2", "Month 3\n(Target)"]
traj = [
    baseline_churn_rate,
    baseline_churn_rate * 0.93,
    baseline_churn_rate * (1 - churn_reduction_pct/100 * 0.7),
    new_churn_rate
]
ax.plot(months, [c*100 for c in traj],
        marker="o", color=PALETTE["accent"], lw=2.5, markersize=9,
        markerfacecolor=PALETTE["warn"], markeredgecolor="white", zorder=3)
ax.fill_between(months, [c*100 for c in traj], alpha=0.15, color=PALETTE["accent"])
ax.axhline(baseline_churn_rate * 100 * 0.85, color=PALETTE["secondary"],
           linestyle="--", lw=1.5, label="−15% target")
ax.set_ylabel("Churn Rate (%)", color="white")
ax.set_title("Projected Churn Reduction Trajectory", color="white")
ax.legend(facecolor="#333", edgecolor="#555", labelcolor="white")
ax.yaxis.grid(True, linestyle="--", alpha=0.2, color="white")
ax.tick_params(colors="white")
for spine in ax.spines.values(): spine.set_color("#444")

# Campaign summary table
ax = fig.add_subplot(gs[2, 0:4])
ax.axis("off")
table = ax.table(
    cellText=[[r["Segment"], r["Strategy"], str(r["Customers"]),
               str(r["Expected Saves"]), r["Campaign Cost"],
               r["Revenue Saved"], r["Net Benefit"], r["ROI %"]]
              for r in campaign_results],
    colLabels=["Segment","Strategy","Customers","Saves",
               "Cost","Revenue Saved","Net Benefit","ROI"],
    loc="center", cellLoc="center"
)
table.auto_set_font_size(False); table.set_fontsize(9); table.scale(1, 2.2)
for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_facecolor(PALETTE["primary"])
        cell.set_text_props(color="white", fontweight="bold")
    else:
        cell.set_facecolor("#2a2a3e" if r % 2 == 0 else "#1a1a2e")
        cell.set_text_props(color="white")
    cell.set_edgecolor("#444")
ax.set_title("Retention Campaign Summary", color="white",
             fontsize=13, fontweight="bold", pad=20)

plt.savefig(OUTPUT_DIR + "4_executive_summary.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✓ 4_executive_summary.png")



# EXPORT SCORED CUSTOMER LIST

print("\n" + "="*60)
print("EXPORT")
print("="*60)

out = df[[
    "CustomerID", "SubscriptionPlan", "AgeGroup", "Location",
    "TotalLogins", "TotalWatchMins", "EngagementScore",
    "Churned", "ChurnProbability", "RiskTier", "SegmentLabel",
    "MonthlyRevenue", "CLV_12M",
]].sort_values("ChurnProbability", ascending=False)

out.to_csv(OUTPUT_DIR + "5_scored_customers.csv", index=False)
print(f"  ✓ 5_scored_customers.csv  ({len(out)} customers)")



# FINAL SUMMARY

print(f"""
{'='*60}
    CHURNLYTICS COMPLETE
{'='*60}
  Dataset          : {len(df)} customers | {len(FEATURES)} features
  Best Model       : {best_name}  (AUC = {best['roc_auc']:.3f})
  Baseline Churn   : {baseline_churn_rate:.2%}
  Customers Saved  : {total_saves} (3-month horizon)
  Churn Reduction  : {churn_reduction_pct:.1f}%  (target 15%)
  Total CLV @ 12M  : ₹{df['CLV_12M'].sum():,.0f}

  Outputs saved to : {OUTPUT_DIR}
    1_eda_dashboard.png
    2_model_evaluation.png
    3_segmentation_dashboard.png
    4_executive_summary.png
    5_scored_customers.csv
{'='*60}
""")


DATA LOADING & MERGING
Activity logs   : (20967, 9)
Demographic     : (250, 4)
Payment history : (250, 4)

FEATURE ENGINEERING
Master dataset  : (250, 19)
Churn rate      : 18.00%

EDA DASHBOARD
  ✓ 1_eda_dashboard.png

ML FEATURE PREPARATION
  Train: (200, 15)  |  Test: (50, 15)
  Churn rate  train=18.00%  test=18.00%

  STEP 5: MODEL TRAINING
  Logistic Regression        ROC-AUC=0.997  F1=0.889  CV=0.996±0.005
  Decision Tree              ROC-AUC=1.000  F1=1.000  CV=0.986±0.029
  Random Forest              ROC-AUC=1.000  F1=1.000  CV=0.992±0.014
  Gradient Boosting          ROC-AUC=1.000  F1=1.000  CV=0.986±0.029

  ★ Best model: Decision Tree  (AUC=1.000)

MODEL EVALUATION DASHBOARD


C:\Users\admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
C:\Users\a

  ✓ 2_model_evaluation.png

RISK SCORING
               Customers  AvgChurnProb  ActualChurned
RiskTier                                             
Critical Risk         45           1.0             45

CUSTOMER SEGMENTATION
        Label  Count  AvgEngagement  AvgChurnProb  Churned
Loyal Engaged     86      71.019520      0.011628        1
    High-Risk     37      27.935968      1.000000       37
Loyal Engaged     83      41.942574      0.000000        0
    Champions     44      86.171472      0.159091        7

SEGMENTATION DASHBOARD
  ✓ 3_segmentation_dashboard.png

RETENTION SIMULATION
  [High-Risk       ]  saves  12 customers  |  ROI = 534%


ValueError: cannot convert float NaN to integer